# Gibbs-Lock and Hamiltonian Extraction — End-to-End Companion

This notebook is the end-to-end companion for the paper  
**"Gibbs-Lock and the Emergence of Hamiltonian Structure in the Inaccessible Game"**  
(implements CIP-000D).

It traces the full chain described in the paper:

1. **Setup** — construct a Gibbs-locked qutrit pair and inspect its spectral geometry  
2. **Loewner kernel and iso-marginal tangency** — classify perturbation modes  
3. **GENERIC decomposition and Hamiltonian extraction** — decompose the constrained flow and extract $H_\text{eff}$  
4. **$\mu_0$ inference** — infer the uniform decay rate from a synthetic trajectory  

**Dependencies:** `qig` package with CIP-000C (`GibbsLockedFrame`, `infer_mu0`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

import qig
from qig import GibbsLockedFrame, loewner_kernel, infer_mu0
from qig.exponential_family import QuantumExponentialFamily
from qig.generic import effective_hamiltonian_coefficients, effective_hamiltonian_operator
from qig.structure_constants import compute_structure_constants
from qig.core import generic_decomposition

print('qig package loaded')
print('GibbsLockedFrame:', GibbsLockedFrame)
print('infer_mu0:', infer_mu0)

---
## Section 1 — Setup: Gibbs-Locked Qutrit Pair

We build a two-qutrit Hamiltonian
$$H = \delta\,(\lambda_3 \otimes I + I \otimes \lambda_3), \qquad \lambda_3 = \operatorname{diag}(1,-1,0),$$
and form the Gibbs-locked frame $K_0 = \beta H$.

Because $[K_0, H] = \beta[H, H] = 0$, the Gibbs-lock condition holds **exactly by construction**.

In [ ]:
delta = 0.5
beta  = 2.0

lam3 = np.array([[1, 0, 0], [0, -1, 0], [0, 0, 0]], dtype=complex)
I3   = np.eye(3, dtype=complex)
H = delta * (np.kron(lam3, I3) + np.kron(I3, lam3))
H -= np.trace(H) / 9 * np.eye(9)

frame = GibbsLockedFrame(H, beta=beta, dims=[3, 3])

print(f'GibbsLockedFrame: D={frame.D}, beta={frame.beta}, dims={frame.dims}')
print(f'Gibbs-lock residual ||[K0,H]||_F = {frame.gibbs_lock_residual():.2e}')

rho0 = frame.rho0
evals_rho0 = np.linalg.eigvalsh(rho0)
print(f'rho0 eigenvalues: {evals_rho0.round(6)}')
print(f'Tr(rho0) = {np.trace(rho0).real:.12f}')

In [ ]:
eps, gaps = frame.bohr_gaps()
print('Eigenvalues of H (Bohr frequencies):', eps.round(4))
print('Unique non-zero |Bohr gaps|:', np.unique(np.abs(gaps[np.abs(gaps) > 1e-10])).round(4))

---
## Section 2 — Loewner Kernel and Iso-Marginal Tangency

The **Loewner divided-difference kernel** maps modular-generator perturbations $\delta K$ to density-matrix perturbations:
$$
(\delta\rho)_{ij} = c(\lambda_i, \lambda_j)\,(\delta K)_{ij},
\qquad
c(\lambda_i, \lambda_j) = \frac{\lambda_i - \lambda_j}{\log\lambda_i - \log\lambda_j}.
$$

A perturbation is **iso-marginal** if $\operatorname{tr}_{\neq k}(J_{\rho_0}(\delta K)) = 0$ for every subsystem $k$.

In [ ]:
C, vals, vecs = frame.loewner_kernel()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(C, cmap='viridis')
ax.set_title('Loewner kernel $C_{ij}$ in eigenbasis of $\\rho_0$')
ax.set_xlabel('$j$'); ax.set_ylabel('$i$')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print(f'Kernel range: [{C.min():.4f}, {C.max():.4f}]')
print(f'LME limit (1/D = 1/9 = {1/9:.4f}); kernel mean = {C.mean():.4f}')

In [ ]:
# Classify perturbation modes
# Mode 1: cross-site off-diagonal coherence (doubly off-diagonal)
delta_K_cross = np.zeros((9, 9), dtype=complex)
delta_K_cross[1, 3] = 1.0   # |01><10| type
delta_K_cross[3, 1] = 1.0

# Mode 2: local perturbation (lambda1 on site 1)
lam1 = np.array([[0, 1, 0], [1, 0, 0], [0, 0, 0]], dtype=complex)
delta_K_local = np.kron(lam1, I3) * 0.1

print('Cross-site off-diagonal mode: iso-marginal =', frame.is_iso_marginal(delta_K_cross))
print('Local site-1 mode:            iso-marginal =', frame.is_iso_marginal(delta_K_local))
print()
print('Note: whether cross-site mode is iso-marginal depends on the specific rho0 eigenstructure.')

---
## Section 3 — GENERIC Decomposition and Hamiltonian Extraction

We instantiate a `QuantumExponentialFamily` for the qutrit pair, perturb slightly away from
the maximally mixed state ($\theta = 0$ corresponds to $\rho = I/9$), and decompose the
constrained-flow Jacobian $M(\theta)$ into symmetric $S$ and antisymmetric $A$ parts.

From $A$ we extract the effective Hamiltonian $H_\text{eff} = \sum_c \eta_c F_c$.

In [ ]:
# Build exponential family for qutrit pair
exp_fam = QuantumExponentialFamily(d=3, n_sites=2)
ops = np.array(exp_fam.operators)   # shape (n_ops, 9, 9)
n_ops = len(ops)
print(f'Exponential family: d=3, n_sites=2, n_ops={n_ops}')

# Work at a small perturbation from the maximally mixed state
# Use theta corresponding to a mix of local and entangling modes
rng = np.random.default_rng(0)
theta_star = rng.standard_normal(n_ops) * 0.3
rho_star = exp_fam.rho_from_theta(theta_star)
print(f'theta_star norm: {np.linalg.norm(theta_star):.4f}')
print(f'rho_star trace: {np.trace(rho_star).real:.6f}')

In [ ]:
# GENERIC decomposition
M = exp_fam.jacobian(theta_star)
S, A = generic_decomposition(M)

print(f'Jacobian shape: {M.shape}')
print(f'||S||_F = {np.linalg.norm(S):.4f}  (dissipative)')
print(f'||A||_F = {np.linalg.norm(A):.4f}  (reversible)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mat, title in zip(axes, [S, A], ['S (symmetric)', 'A (antisymmetric)']):
    im = ax.imshow(mat, cmap='RdBu_r')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.suptitle('GENERIC decomposition at $\\theta^*$')
plt.tight_layout()
plt.show()

In [ ]:
# Extract effective Hamiltonian from antisymmetric sector
# ops_list is the original list; ops is the array form for structure constants
ops_list = exp_fam.operators
f_abc = compute_structure_constants(ops)  # ops is np.array(exp_fam.operators)
# effective_hamiltonian_coefficients returns (eta, diagnostics_dict)
eta, extraction_info = effective_hamiltonian_coefficients(A, theta_star, f_abc)
H_eff = effective_hamiltonian_operator(eta, ops_list)  # needs list of matrices

print(f'H_eff Hermitian: {np.allclose(H_eff, H_eff.conj().T, atol=1e-8)}')
print(f'H_eff traceless: {abs(np.trace(H_eff)) < 1e-8}')
print(f'H_eff Frobenius norm: {np.linalg.norm(H_eff):.4f}')

# Verification: compare antisymmetric flow with commutator bracket
# The parameter-space antisymmetric flow induces rho flow via Kubo-Mori derivatives
param_flow = A @ theta_star
drho_rev = sum(param_flow[a] * exp_fam.rho_derivative(theta_star, a) for a in range(n_ops))
drho_comm = -1j * (H_eff @ rho_star - rho_star @ H_eff)

norm_rev = np.linalg.norm(drho_rev, 'fro')
norm_comm = np.linalg.norm(drho_comm, 'fro')
print(f'\n||d_rho_rev||    = {norm_rev:.4f}')
print(f'||-i[H_eff,rho]|| = {norm_comm:.4f}')
print()
print('Note: these norms differ because the Kubo-Mori inner product != commutator.')
print('See docs/source/theory/hamiltonian_extraction.rst for the theoretical context.')

---
## Section 4 — $\mu_0$ Inference and Resolution Floor

The uniform decay rate $\mu_0$ governs off-diagonal coherence decay:
$$|(\delta\rho)_{ij}(t)| = |(\delta\rho)_{ij}(0)|\, e^{-\mu_0 t}.$$

We build a synthetic trajectory from `GibbsLockedFrame.linearised_flow` (exact analytical solution)
and recover $\mu_0$ via `infer_mu0`.  This links to **Experiment 5** of
`hamiltonian_emergence_experiments.ipynb`, which connects $\mu_0$ to the Fisher-information
resolution floor.

In [ ]:
mu0_true = 0.4
n_points = 60
t_max = 5.0
times = np.linspace(0.0, t_max, n_points)

# Build perturbation directly in the H eigenbasis (ensures visible magnitudes)
_, H_vecs = frame._eigh_H()
_, gaps_mat = frame.bohr_gaps()

rng = np.random.default_rng(42)
dK0_eig = (rng.standard_normal((9, 9)) + 1j * rng.standard_normal((9, 9))) * 0.05
dK0_eig = (dK0_eig + dK0_eig.conj().T) / 2
# Ensure all off-diagonal amplitudes exceed the tol=1e-3 threshold
for i in range(9):
    for j in range(9):
        if i != j and abs(dK0_eig[i, j]) < 0.02:
            dK0_eig[i, j] = 0.025 + 0j

# Analytical trajectory: element-wise exp decay + phase rotation
rho_traj = np.zeros((n_points, 9, 9), dtype=complex)
for ti, t in enumerate(times):
    phase = np.exp((1j * beta * gaps_mat - mu0_true) * t)
    dR_eig_t = dK0_eig * phase
    delta_rho = H_vecs @ dR_eig_t @ H_vecs.conj().T
    rho_traj[ti] = rho0 + delta_rho

print(f'Trajectory built: {n_points} steps, t in [0, {t_max}]')

In [ ]:
mu0_fit = infer_mu0(times, rho_traj, frame)
print(f'True  mu_0         = {mu0_true:.4f}')
print(f'Inferred mu_0      = {mu0_fit:.4f}')
print(f'Relative error     = {abs(mu0_fit - mu0_true)/mu0_true*100:.2f}%')

In [ ]:
# Plot: mean off-diagonal decay vs exponential fits
delta_rho_eig = np.array([
    H_vecs.conj().T @ (rho_traj[ti] - rho0) @ H_vecs
    for ti in range(n_points)
])

mags = []
for i in range(9):
    for j in range(9):
        if i != j:
            m = np.abs(delta_rho_eig[:, i, j])
            if m[0] > 1e-3:
                mags.append(m / m[0])

mean_decay = np.mean(mags, axis=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(times, mean_decay, 'b-', lw=1.5, alpha=0.7, label='Mean off-diagonal magnitude')
ax.semilogy(times, np.exp(-mu0_true * times), 'r--', lw=2,
            label=f'True $e^{{-\\mu_0 t}}$, $\\mu_0={mu0_true}$')
ax.semilogy(times, np.exp(-mu0_fit * times), 'g:', lw=2,
            label=f'Fitted $e^{{-\\hat\\mu_0 t}}$, $\\hat\\mu_0={mu0_fit:.3f}$')
ax.set_xlabel('Time $t$')
ax.set_ylabel('Normalised off-diagonal magnitude')
ax.set_title('Uniform off-diagonal decay: true vs inferred $\\mu_0$')
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary

| Step | Object / function | Key result |
|------|-------------------|------------|
| 1 | `GibbsLockedFrame(H, beta, dims=[3,3])` | $\|[K_0, H]\|_F < 10^{-14}$ |
| 1 | `frame.bohr_gaps()` | Bohr frequencies from spec$(H)$ |
| 2 | `frame.loewner_kernel()` | Divided-difference kernel $C_{ij}$ |
| 2 | `frame.is_iso_marginal(\delta K)` | Mode classification |
| 3 | `effective_hamiltonian_operator` | $H_\text{eff}$ Hermitian + traceless |
| 4 | `infer_mu0(times, rho_traj, frame)` | $\hat\mu_0$ within 5% of true |

### Cross-references

- `examples/hamiltonian_emergence_experiments.ipynb` — structural claim validation (CIP-000B)  
- `docs/source/theory/hamiltonian_extraction.rst` — Gibbs-lock theory and extraction algorithm  
- `qig.gibbs_lock` module documentation — `GibbsLockedFrame`, `infer_mu0`  
- CIP-000C and CIP-000D